In [1]:
# Setup and mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Install required packages
import sys
!{sys.executable} -m pip install -q roboflow albumentations

import os
import json
import shutil
from pathlib import Path
from collections import Counter, defaultdict
from roboflow import Roboflow

print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 73.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2025.3.0 requires fsspec==2025.3.0.*, but you have fsspec 2025.10.0 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2

In [2]:
# Download all 10 datasets from Roboflow
print("Downloading datasets from Roboflow...")

# Initialize Roboflow API
rf = Roboflow(api_key="0Wgp8GjEQCPnJbk4eVeF")

# Define all 10 project configurations
PROJECTS = [
    ('cans-avukd-b09pf', 3, 'cans'),
    ('fire-8klur-fceqf', 2, 'fire'),
    ('traffic-light-nugkt-b6lro', 2, 'traffic_light'),
    ('pothole-vhmow-smavg', 2, 'pothole'),
    ('helmet-motorcycle-detection-pro0q-ozzzn', 2, 'helmet'),
    ('pen-kkqzs-tvftk', 2, 'pen'),
    ('e-waste-mobile-phones-drlac', 2, 'e_waste'),
    ('bags-6zumh-aduq2', 3, 'bags'),
    ('water-bottle-qginj-6glhq', 3, 'water_bottle'),
    ('knifedetectionv4-6pfwh', 2, 'knife'),
]

downloaded_datasets = []

# Download each dataset
for idx, (project_id, version, class_name) in enumerate(PROJECTS, 1):
    try:
        print(f"[{idx}/10] Downloading {class_name}...")
        project = rf.workspace("yuyutsu-cxcso").project(project_id)
        dataset = project.version(version).download("coco")

        downloaded_datasets.append({
            'name': class_name,
            'path': dataset.location,
            'class_id': idx - 1
        })

        print(f"    Downloaded to: {dataset.location}")
    except Exception as e:
        print(f"    Error: {e}")

print(f"\nDownloaded {len(downloaded_datasets)}/10 datasets")
print("\nClass mapping:")
for ds in downloaded_datasets:
    print(f"  Class {ds['class_id']}: {ds['name']}")

[1/10] Downloading cans...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to cans-3 in coco:: 100%|██████████| 7060/7060 [00:01<00:00, 4023.81it/s]


    Downloaded to: /kaggle/working/cans-3
[2/10] Downloading fire...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to fire-2 in coco:: 100%|██████████| 6105/6105 [00:00<00:00, 6719.13it/s]


    Downloaded to: /kaggle/working/fire-2
[3/10] Downloading traffic_light...
loading Roboflow workspace...
loading Roboflow project...


...


Extracting Dataset Version Zip to Water-bottle-3 in coco:: 100%|██████████| 2522/2522 [00:00<00:00, 7069.59it/s]

    Downloaded to: /kaggle/working/Water-bottle-3
[10/10] Downloading knife...
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to KnifeDetectionV4-2 in coco:: 100%|██████████| 2607/2607 [00:00<00:00, 7359.85it/s]

    Downloaded to: /kaggle/working/KnifeDetectionV4-2

Downloaded 10/10 datasets

Class mapping:
  Class 0: cans
  Class 1: fire
  Class 2: traffic_light
  Class 3: pothole
  Class 4: helmet
  Class 5: pen
  Class 6: e_waste
  Class 7: bags
  Class 8: water_bottle
  Class 9: knife


In [3]:
# Merge all datasets into one unified COCO dataset
def merge_coco_datasets(datasets_info, output_dir='/kaggle/working/FYP_10Class_Dataset'):
    print("\nMerging all datasets...")

    # Create output directories
    os.makedirs(output_dir, exist_ok=True)
    for split in ['train', 'valid', 'test']:
        os.makedirs(os.path.join(output_dir, split), exist_ok=True)

    # Merge each split separately
    for split in ['train', 'valid', 'test']:
        print(f"\nProcessing {split} split...")

        merged_images = []
        merged_annotations = []
        image_id_counter = 1
        annotation_id_counter = 1

        # Process each dataset
        for dataset_info in datasets_info:
            dataset_name = dataset_info['name']
            dataset_path = dataset_info['path']
            class_id = dataset_info['class_id']

            # Load annotations
            split_path = os.path.join(dataset_path, split)
            json_path = os.path.join(split_path, '_annotations.coco.json')

            if not os.path.exists(json_path):
                print(f"  Warning: {dataset_name}/{split} not found, skipping")
                continue

            with open(json_path, 'r') as f:
                coco_data = json.load(f)

            # Map old image IDs to new ones
            old_to_new_image_id = {}

            # Process images
            for img in coco_data['images']:
                old_img_id = img['id']
                new_img_id = image_id_counter
                old_to_new_image_id[old_img_id] = new_img_id
                image_id_counter += 1

                # Copy image with unique name
                old_img_path = os.path.join(split_path, img['file_name'])
                file_ext = os.path.splitext(img['file_name'])[1]
                new_img_name = f"{dataset_name}_{new_img_id}{file_ext}"
                new_img_path = os.path.join(output_dir, split, new_img_name)

                if os.path.exists(old_img_path):
                    shutil.copy(old_img_path, new_img_path)

                merged_images.append({
                    'id': new_img_id,
                    'file_name': new_img_name,
                    'width': img.get('width', 640),
                    'height': img.get('height', 640)
                })

            # Process annotations
            ann_count = 0
            for ann in coco_data['annotations']:
                if ann['image_id'] not in old_to_new_image_id:
                    continue

                new_img_id = old_to_new_image_id[ann['image_id']]

                merged_annotations.append({
                    'id': annotation_id_counter,
                    'image_id': new_img_id,
                    'category_id': class_id,
                    'bbox': ann['bbox'],
                    'area': ann.get('area', ann['bbox'][2] * ann['bbox'][3]),
                    'iscrowd': ann.get('iscrowd', 0)
                })

                annotation_id_counter += 1
                ann_count += 1

            img_count = len(coco_data['images'])
            print(f"  {dataset_name}: {img_count} images, {ann_count} annotations")

        # Save merged annotations
        merged_coco = {
            'images': merged_images,
            'annotations': merged_annotations,
            'categories': [
                {'id': ds['class_id'], 'name': ds['name'], 'supercategory': 'object'}
                for ds in datasets_info
            ]
        }

        output_json = os.path.join(output_dir, split, '_annotations.coco.json')
        with open(output_json, 'w') as f:
            json.dump(merged_coco, f, indent=2)

        print(f"  {split} total: {len(merged_images)} images, {len(merged_annotations)} annotations")

    print(f"\nMerge complete. Saved to: {output_dir}")
    return output_dir

# Run merge
merged_dataset_path = merge_coco_datasets(downloaded_datasets)
print("\nDataset saved permanently at /kaggle/working/FYP_10Class_Dataset")
print("Ready for training on Kaggle GPU")


Merging all datasets...

Processing train split...
  cans: 6579 images, 6645 annotations
  fire: 5349 images, 14115 annotations
  traffic_light: 2973 images, 3616 annotations
  pothole: 3840 images, 6440 annotations
  helmet: 2665 images, 6575 annotations
  pen: 5240 images, 7315 annotations
  e_waste: 2424 images, 2831 annotations
  bags: 1112 images, 1267 annotations
  water_bottle: 2286 images, 3543 annotations
  knife: 2311 images, 2390 annotations
  train total: 34779 images, 54737 annotations

Processing valid split...
  cans: 290 images, 293 annotations
  fire: 363 images, 896 annotations
  traffic_light: 158 images, 184 annotations
  helmet: 114 images, 130 annotations
  pen: 506 images, 706 annotations
  e_waste: 406 images, 458 annotations
  bags: 107 images, 133 annotations
  water_bottle: 178 images, 329 annotations
  knife: 181 images, 187 annotations
  valid total: 2303 images, 3316 annotations

Processing test split...
  cans: 183 images, 185 annotations
  fire: 385 ima

In [4]:
# Import required libraries for data processing
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
import matplotlib.pyplot as plt
from glob import glob
import albumentations as A
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Configuration class with all parameters
class Config:
    DATASET_PATH = '/kaggle/working/FYP_10Class_Dataset'     # ← ONLY CHANGE 2
    INPUT_SIZE = 640
    GRID_SIZE = 20
    NUM_CLASSES = 10
    NUM_ANCHORS = 3
    BATCH_SIZE = 16
    EPOCHS = 150
    LEARNING_RATE = 5e-5
    CLASS_NAMES = [
        'cans', 'fire', 'traffic_light', 'pothole', 'helmet',
        'pen', 'e_waste', 'bags', 'water_bottle', 'knife'
    ]
    CLASS_WEIGHTS = {
        0: 2.12, 1: 1.0, 2: 3.90, 3: 2.19, 4: 2.15,
        5: 1.93, 6: 4.99, 7: 11.14, 8: 3.98, 9: 5.91,
    }
    ANCHORS = [
        (30, 40), (50, 60), (40, 80),
        (100, 120), (150, 180), (120, 200),
        (250, 300), (350, 400), (500, 550)
    ]

cfg = Config()
print(f"Configuration loaded. Learning rate: {cfg.LEARNING_RATE}")

2025-12-12 14:48:23.897329: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765550904.625233      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765550904.747096      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version: 2.18.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Configuration loaded. Learning rate: 5e-05


In [6]:
# Function to clip bounding boxes to image boundaries
def clip_bboxes(bboxes, img_w=640, img_h=640):
    if len(bboxes) == 0:
        return []

    clipped = []
    for bbox in bboxes:
        x, y, w, h = bbox[:4]

        # Clip to image bounds
        x = np.clip(x, 0, img_w - 1)
        y = np.clip(y, 0, img_h - 1)
        w = np.clip(w, 1, img_w - x)
        h = np.clip(h, 1, img_h - y)

        if w > 0 and h > 0:
            clipped_bbox = [x, y, w, h]
            if len(bbox) > 4:
                clipped_bbox.extend(bbox[4:])
            clipped.append(clipped_bbox)

    return clipped

# Setup data augmentation for training
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=5,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.4
    ),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['class_labels'],
    min_visibility=0.3,
    clip=True
))

# No augmentation for validation
val_transform = A.Compose([
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['class_labels'],
    clip=True
))

print("Data augmentation configured")

Data augmentation configured


Argument(s) 'value' are not valid for transform ShiftScaleRotate
Argument(s) 'var_limit' are not valid for transform GaussNoise


In [7]:
# Function to load COCO format annotations
def load_coco_annotations(json_path, img_dir):
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # Create mapping from image ID to filename
    id_to_file = {img['id']: img['file_name'] for img in coco_data['images']}

    # Group annotations by image
    annotations_by_image = {}
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

    # Create dataset records
    records = []
    for img_id, anns in annotations_by_image.items():
        if img_id not in id_to_file:
            continue

        img_path = os.path.join(img_dir, id_to_file[img_id])
        if not os.path.exists(img_path):
            continue

        boxes = []
        classes = []

        for ann in anns:
            x, y, w, h = ann['bbox']
            if w > 0 and h > 0:
                boxes.append([float(x), float(y), float(w), float(h)])
                classes.append(int(ann['category_id']))

        if len(boxes) > 0:
            records.append({
                'image_path': img_path,
                'boxes': np.array(boxes, dtype=np.float32),
                'classes': np.array(classes, dtype=np.int32)
            })

    return records

print("Annotation loader ready")

Annotation loader ready


In [8]:
# Function to encode bounding boxes to grid format
def encode_to_grid(boxes, classes, grid_size=20, input_size=640, num_classes=10):
    num_anchors = 3
    grid = np.zeros((grid_size, grid_size, num_anchors, 5 + num_classes), dtype=np.float32)

    # Convert to numpy arrays if needed
    if not isinstance(boxes, np.ndarray):
        boxes = np.array(boxes, dtype=np.float32)
    if not isinstance(classes, np.ndarray):
        classes = np.array(classes, dtype=np.int32)

    if len(boxes) == 0:
        return grid

    cell_size = input_size / grid_size
    anchors = np.array(cfg.ANCHORS[:3], dtype=np.float32)

    for box, cls in zip(boxes, classes):
        if isinstance(box, (tuple, list)):
            box = np.array(box, dtype=np.float32)

        x, y, w, h = float(box[0]), float(box[1]), float(box[2]), float(box[3])

        if w <= 0 or h <= 0:
            continue

        # Convert to center coordinates
        cx = x + w / 2
        cy = y + h / 2

        # Find grid cell
        grid_x = int(np.clip(cx / cell_size, 0, grid_size - 1))
        grid_y = int(np.clip(cy / cell_size, 0, grid_size - 1))

        # Find best matching anchor
        best_anchor = 0
        best_iou = 0

        for anchor_idx in range(num_anchors):
            anchor_w, anchor_h = anchors[anchor_idx]
            intersect_w = min(w, anchor_w)
            intersect_h = min(h, anchor_h)
            intersect = intersect_w * intersect_h
            union = w * h + anchor_w * anchor_h - intersect
            iou = intersect / (union + 1e-6)

            if iou > best_iou:
                best_iou = iou
                best_anchor = anchor_idx

        # Encode box relative to grid cell
        x_cell = (cx % cell_size) / cell_size
        y_cell = (cy % cell_size) / cell_size
        w_anchor = np.log((w + 1e-6) / (anchors[best_anchor][0] + 1e-6))
        w_anchor = np.clip(w_anchor, -10, 10)
        h_anchor = np.log((h + 1e-6) / (anchors[best_anchor][1] + 1e-6))
        h_anchor = np.clip(h_anchor, -10, 10)

        cls = int(cls)
        if cls < 0 or cls >= num_classes:
            continue

        # Encode to grid if cell not occupied
        if grid[grid_y, grid_x, best_anchor, 4] == 0:
            grid[grid_y, grid_x, best_anchor, 0] = x_cell
            grid[grid_y, grid_x, best_anchor, 1] = y_cell
            grid[grid_y, grid_x, best_anchor, 2] = w_anchor
            grid[grid_y, grid_x, best_anchor, 3] = h_anchor
            grid[grid_y, grid_x, best_anchor, 4] = 1.0
            grid[grid_y, grid_x, best_anchor, 5 + cls] = 1.0

    return grid

print("Grid encoder ready")

Grid encoder ready


In [9]:
# Function to preprocess images and boxes
def preprocess_image_and_boxes(record, transform, training=True):
    try:
        # Load image
        image = cv2.imread(record['image_path'])
        if image is None:
            raise ValueError(f"Failed to load image")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h_orig, w_orig = image.shape[:2]

        # Get boxes and classes
        boxes = record['boxes'].tolist() if isinstance(record['boxes'], np.ndarray) else record['boxes']
        classes = record['classes'].tolist() if isinstance(record['classes'], np.ndarray) else record['classes']

        # Apply augmentation
        if training and transform is not None:
            try:
                transformed = transform(image=image, bboxes=boxes, class_labels=classes)
                image = transformed['image']
                boxes = transformed['bboxes']
                classes = transformed['class_labels']
            except:
                pass

        # Clip boxes before resize
        if len(boxes) > 0:
            boxes = clip_bboxes(boxes, image.shape[1], image.shape[0])

        # Convert to numpy
        if len(boxes) > 0:
            boxes = np.array(boxes, dtype=np.float32)
            classes = np.array(classes, dtype=np.int32)
        else:
            boxes = np.zeros((0, 4), dtype=np.float32)
            classes = np.array([], dtype=np.int32)

        # Resize image
        image = cv2.resize(image, (cfg.INPUT_SIZE, cfg.INPUT_SIZE))

        # Scale boxes to resized image
        if len(boxes) > 0:
            scale_x = cfg.INPUT_SIZE / w_orig
            scale_y = cfg.INPUT_SIZE / h_orig

            boxes[:, 0] *= scale_x
            boxes[:, 1] *= scale_y
            boxes[:, 2] *= scale_x
            boxes[:, 3] *= scale_y

            # Clip boxes after resize
            boxes = np.array(clip_bboxes(boxes, cfg.INPUT_SIZE, cfg.INPUT_SIZE), dtype=np.float32)

            if len(boxes) > 0:
                boxes[:, 0] = np.clip(boxes[:, 0], 0, cfg.INPUT_SIZE - 1)
                boxes[:, 1] = np.clip(boxes[:, 1], 0, cfg.INPUT_SIZE - 1)
                boxes[:, 2] = np.clip(boxes[:, 2], 1, cfg.INPUT_SIZE - boxes[:, 0])
                boxes[:, 3] = np.clip(boxes[:, 3], 1, cfg.INPUT_SIZE - boxes[:, 1])

        # Normalize image
        image = image.astype(np.float32) / 255.0

        # Check for invalid data
        if np.any(np.isnan(image)) or np.any(np.isinf(image)):
            raise ValueError("Invalid image data")

        # Encode to grid
        grid = encode_to_grid(boxes, classes, cfg.GRID_SIZE, cfg.INPUT_SIZE, cfg.NUM_CLASSES)

        if np.any(np.isnan(grid)) or np.any(np.isinf(grid)):
            raise ValueError("Invalid grid data")

        return image, grid

    except:
        # Return empty data on error
        image = np.zeros((cfg.INPUT_SIZE, cfg.INPUT_SIZE, 3), dtype=np.float32)
        grid = np.zeros((cfg.GRID_SIZE, cfg.GRID_SIZE, 3, 5 + cfg.NUM_CLASSES), dtype=np.float32)
        return image, grid

print("Preprocessing function ready")

Preprocessing function ready


In [10]:
# Function to create TensorFlow dataset
def create_dataset(split='train', batch_size=8, shuffle=True):
    json_path = os.path.join(cfg.DATASET_PATH, split, '_annotations.coco.json')
    img_dir = os.path.join(cfg.DATASET_PATH, split)

    print(f"Loading {split} split...")
    records = load_coco_annotations(json_path, img_dir)
    print(f"  Loaded {len(records)} images")

    transform = train_transform if split == 'train' else val_transform
    training = (split == 'train')

    def generator():
        indices = list(range(len(records)))
        if shuffle:
            np.random.shuffle(indices)

        for idx in indices:
            try:
                image, grid = preprocess_image_and_boxes(records[idx], transform, training)
                if not (np.isnan(image).any() or np.isnan(grid).any()):
                    yield image, grid
            except:
                continue

    output_signature = (
        tf.TensorSpec(shape=(cfg.INPUT_SIZE, cfg.INPUT_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(cfg.GRID_SIZE, cfg.GRID_SIZE, 3, 5 + cfg.NUM_CLASSES), dtype=tf.float32)
    )

    dataset = tf.data.Dataset.from_generator(generator, output_signature=output_signature)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, len(records)

# Create train and validation datasets
print("\nCreating datasets...")
train_dataset, train_size = create_dataset('train', batch_size=cfg.BATCH_SIZE, shuffle=True)
valid_dataset, valid_size = create_dataset('valid', batch_size=cfg.BATCH_SIZE, shuffle=False)

print(f"\nDatasets ready:")
print(f"  Train: {train_size} images ({train_size // cfg.BATCH_SIZE} steps/epoch)")
print(f"  Valid: {valid_size} images ({valid_size // cfg.BATCH_SIZE} steps/epoch)")

# Verify datasets
for images, grids in train_dataset.take(1):
    print(f"\nVerification:")
    print(f"  Image batch: {images.shape}")
    print(f"  Grid batch: {grids.shape}")
    print(f"  Value range: [{images.numpy().min():.3f}, {images.numpy().max():.3f}]")

print("\nData pipeline complete")


Creating datasets...
Loading train split...
  Loaded 33448 images


I0000 00:00:1765550984.905438      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1765550984.906267      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Loading valid split...
  Loaded 2300 images

Datasets ready:
  Train: 33448 images (2090 steps/epoch)
  Valid: 2300 images (143 steps/epoch)

Verification:
  Image batch: (16, 640, 640, 3)
  Grid batch: (16, 20, 20, 3, 15)
  Value range: [0.000, 1.000]


Exception ignored in: <generator object create_dataset.<locals>.generator at 0x7f485f9a6440>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/dataset_ops.py", line 872, in iterator_completed
    del self._iterators[self._normalize_id(iterator_id)]
        ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: generator ignored GeneratorExit



Data pipeline complete


In [12]:
# Build custom object detection model
from tensorflow.keras import layers, Model

def conv_block(x, filters, kernel_size=3, strides=1, activation='leaky_relu', batch_norm=True, name=''):
    x = layers.Conv2D(
        filters, kernel_size, strides=strides, padding='same',
        use_bias=not batch_norm, kernel_initializer='he_normal',
        name=f'{name}_conv' if name else None
    )(x)

    if batch_norm:
        x = layers.BatchNormalization(name=f'{name}_bn' if name else None)(x)

    if activation == 'leaky_relu':
        x = layers.LeakyReLU(alpha=0.1, name=f'{name}_act' if name else None)(x)

    return x

def residual_block(x, filters, name=''):
    shortcut = x
    x = conv_block(x, filters // 2, kernel_size=1, name=f'{name}_conv1')
    x = conv_block(x, filters, kernel_size=3, name=f'{name}_conv2')
    x = layers.Add(name=f'{name}_add')([shortcut, x])
    return x

def build_detector_model(input_shape=(640, 640, 3), num_classes=10, num_anchors=3):
    inputs = layers.Input(shape=input_shape, name='input')

    print("\nBuilding model...")

    # Backbone - feature extraction
    x = conv_block(inputs, 32, kernel_size=3, strides=1, name='stem')

    x = conv_block(x, 64, kernel_size=3, strides=2, name='stage1_down')
    x = residual_block(x, 64, name='stage1_res1')

    x = conv_block(x, 128, kernel_size=3, strides=2, name='stage2_down')
    x = residual_block(x, 128, name='stage2_res1')
    x = residual_block(x, 128, name='stage2_res2')

    x = conv_block(x, 256, kernel_size=3, strides=2, name='stage3_down')
    x = residual_block(x, 256, name='stage3_res1')
    x = residual_block(x, 256, name='stage3_res2')
    x = residual_block(x, 256, name='stage3_res3')

    x = conv_block(x, 512, kernel_size=3, strides=2, name='stage4_down')
    x = residual_block(x, 512, name='stage4_res1')
    x = residual_block(x, 512, name='stage4_res2')
    x = residual_block(x, 512, name='stage4_res3')

    x = conv_block(x, 512, kernel_size=3, strides=2, name='stage5_down')
    x = residual_block(x, 512, name='stage5_res1')
    x = residual_block(x, 512, name='stage5_res2')

    # Neck - feature refinement
    x = conv_block(x, 512, kernel_size=3, name='neck_conv1')
    x = conv_block(x, 512, kernel_size=1, name='neck_conv2')
    x = conv_block(x, 512, kernel_size=3, name='neck_conv3')

    # Detection head
    num_outputs = num_anchors * (5 + num_classes)
    x = conv_block(x, 256, kernel_size=1, name='head_conv1')
    x = conv_block(x, 512, kernel_size=3, name='head_conv2')
    output = layers.Conv2D(num_outputs, kernel_size=1, padding='same',
                          kernel_initializer='glorot_uniform', name='output')(x)

    # Reshape to grid format
    output = layers.Reshape((20, 20, num_anchors, 5 + num_classes), name='reshape')(output)

    model = Model(inputs=inputs, outputs=output, name='ObjectDetector')

    print("Model built successfully")
    return model

# Create model
print("\nInitializing model...")
model = build_detector_model(
    input_shape=(cfg.INPUT_SIZE, cfg.INPUT_SIZE, 3),
    num_classes=cfg.NUM_CLASSES,
    num_anchors=3
)

# Show model summary
model.summary()

total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")
print(f"Model size: {(total_params * 4) / (1024 * 1024):.2f} MB")

# Test forward pass
print("\nTesting forward pass...")
dummy_input = tf.random.normal((1, 640, 640, 3))
dummy_output = model(dummy_input, training=False)
print(f"  Input: {dummy_input.shape}")
print(f"  Output: {dummy_output.shape}")
print("\nModel ready for training")

# REPLACE YOUR ENTIRE MODEL BUILDING CELL WITH THIS ONE
# import tensorflow as tf
# from tensorflow.keras import layers, Model

# # ACTIVATE BOTH T4 GPUs
# strategy = tf.distribute.MirroredStrategy()
# print("GPUs ACTIVE →", strategy.num_replicas_in_sync)   # ← MUST SHOW 2

# with strategy.scope():
#     def conv_block(x, filters, kernel_size=3, strides=1, activation='leaky_relu', batch_norm=True, name=''):
#         x = layers.Conv2D(
#             filters, kernel_size, strides=strides, padding='same',
#             use_bias=not batch_norm, kernel_initializer='he_normal',
#             name=f'{name}_conv' if name else None
#         )(x)
#         if batch_norm:
#             x = layers.BatchNormalization(name=f'{name}_bn' if name else None)(x)
#         if activation == 'leaky_relu':
#             x = layers.LeakyReLU(alpha=0.1, name=f'{name}_act' if name else None)(x)
#         return x

#     def residual_block(x, filters, name=''):
#         shortcut = x
#         x = conv_block(x, filters // 2, kernel_size=1, name=f'{name}_conv1')
#         x = conv_block(x, filters, kernel_size=3, name=f'{name}_conv2')
#         x = layers.Add(name=f'{name}_add')([shortcut, x])
#         return x

#     def build_detector_model(input_shape=(640, 640, 3), num_classes=10, num_anchors=3):
#         inputs = layers.Input(shape=input_shape, name='input')
#         print("\nBuilding model...")
        
#         x = conv_block(inputs, 32, kernel_size=3, strides=1, name='stem')
#         x = conv_block(x, 64, kernel_size=3, strides=2, name='stage1_down')
#         x = residual_block(x, 64, name='stage1_res1')
#         x = conv_block(x, 128, kernel_size=3, strides=2, name='stage2_down')
#         x = residual_block(x, 128, name='stage2_res1')
#         x = residual_block(x, 128, name='stage2_res2')
#         x = conv_block(x, 256, kernel_size=3, strides=2, name='stage3_down')
#         x = residual_block(x, 256, name='stage3_res1')
#         x = residual_block(x, 256, name='stage3_res2')
#         x = residual_block(x, 256, name='stage3_res3')
#         x = conv_block(x, 512, kernel_size=3, strides=2, name='stage4_down')
#         x = residual_block(x, 512, name='stage4_res1')
#         x = residual_block(x, 512, name='stage4_res2')
#         x = residual_block(x, 512, name='stage4_res3')
#         x = conv_block(x, 512, kernel_size=3, strides=2, name='stage5_down')
#         x = residual_block(x, 512, name='stage5_res1')
#         x = residual_block(x, 512, name='stage5_res2')
        
#         x = conv_block(x, 512, kernel_size=3, name='neck_conv1')
#         x = conv_block(x, 512, kernel_size=1, name='neck_conv2')
#         x = conv_block(x, 512, kernel_size=3, name='neck_conv3')
        
#         num_outputs = num_anchors * (5 + num_classes)
#         x = conv_block(x, 256, kernel_size=1, name='head_conv1')
#         x = conv_block(x, 512, kernel_size=3, name='head_conv2')
#         output = layers.Conv2D(num_outputs, kernel_size=1, padding='same',
#                               kernel_initializer='glorot_uniform', name='output')(x)
#         output = layers.Reshape((20, 20, num_anchors, 5 + num_classes), name='reshape')(output)
        
#         model = Model(inputs=inputs, outputs=output, name='ObjectDetector')
#         print("Model built successfully")
#         return model

#     # BUILD MODEL ON GPU
#     model = build_detector_model(
#         input_shape=(cfg.INPUT_SIZE, cfg.INPUT_SIZE, 3),
#         num_classes=cfg.NUM_CLASSES,
#         num_anchors=3
#     )

#     # YOUR LOSS
#     loss_fn = DetectionLoss(
#         num_classes=cfg.NUM_CLASSES,
#         lambda_coord=5.0,
#         lambda_obj=1.0,
#         lambda_class=1.0,
#         lambda_noobj=0.5,
#         class_weights=cfg.CLASS_WEIGHTS
#     )

#     # 2× LEARNING RATE FOR 2 GPUS
#     optimizer = tf.keras.optimizers.Adam(
#         learning_rate=cfg.LEARNING_RATE * 2,   # 5e-5 → 1e-4
#         clipnorm=5.0,
#         epsilon=1e-7
#     )

#     model.compile(optimizer=optimizer, loss=loss_fn)

# print("MODEL IS NOW USING BOTH T4 GPUS — TRAINING WILL BE FAST")
# model.summary()


Initializing model...

Building model...
Model built successfully


Model: "ObjectDetector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 640, 640,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 640, 640,  │        864 │ input[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 640, 640,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_act            │ (None, 640, 640,  │          0 │ stem_bn[0][0]     │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_down_conv    │ (None, 320, 320,  │     18,432 │ stem_act[0][0]    │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_down_bn      │ (None, 320, 320,  │        256 │ stage1_down_conv… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_down_act     │ (None, 320, 320,  │          0 │ stage1_down_bn[0… │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv1_… │ (None, 320, 320,  │      2,048 │ stage1_down_act[… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv1_… │ (None, 320, 320,  │        128 │ stage1_res1_conv… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv1_… │ (None, 320, 320,  │          0 │ stage1_res1_conv… │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv2_… │ (None, 320, 320,  │     18,432 │ stage1_res1_conv… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv2_… │ (None, 320, 320,  │        256 │ stage1_res1_conv… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_conv2_… │ (None, 320, 320,  │          0 │ stage1_res1_conv… │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_res1_add     │ (None, 320, 320,  │          0 │ stage1_down_act[… │
│ (Add)               │ 64)               │            │ stage1_res1_conv… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_down_conv    │ (None, 160, 160,  │     73,728 │ stage1_res1_add[… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_down_bn      │ (None, 160, 160,  │        512 │ stage2_down_conv… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_down_act     │ (None, 160, 160,  │          0 │ stage2_down_bn[0

 Total params: 17,999,501 (68.66 MB)

 Trainable params: 17,980,941 (68.59 MB)

 Non-trainable params: 18,560 (72.50 KB)


Total parameters: 17,999,501
Model size: 68.66 MB

Testing forward pass...


I0000 00:00:1765551053.350819      47 cuda_dnn.cc:529] Loaded cuDNN version 90300


  Input: (1, 640, 640, 3)
  Output: (1, 20, 20, 3, 15)

Model ready for training


In [13]:
def compute_iou(box1, box2):
    # Convert center format -> corner format
    box1_x1 = box1[..., 0] - box1[..., 2] / 2
    box1_y1 = box1[..., 1] - box1[..., 3] / 2
    box1_x2 = box1[..., 0] + box1[..., 2] / 2
    box1_y2 = box1[..., 1] + box1[..., 3] / 2

    box2_x1 = box2[..., 0] - box2[..., 2] / 2
    box2_y1 = box2[..., 1] - box2[..., 3] / 2
    box2_x2 = box2[..., 0] + box2[..., 2] / 2
    box2_y2 = box2[..., 1] + box2[..., 3] / 2

    # Intersection
    inter_x1 = tf.maximum(box1_x1, box2_x1)
    inter_y1 = tf.maximum(box1_y1, box2_y1)
    inter_x2 = tf.minimum(box1_x2, box2_x2)
    inter_y2 = tf.minimum(box1_y2, box2_y2)

    inter_w = tf.maximum(0.0, inter_x2 - inter_x1)
    inter_h = tf.maximum(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    # Union
    box1_area = box1[..., 2] * box1[..., 3]
    box2_area = box2[..., 2] * box2[..., 3]
    union_area = box1_area + box2_area - inter_area

    # IoU
    iou = inter_area / (union_area + 1e-7)
    return iou


# ---------------------
# GIoU
# ---------------------
def compute_giou(box1, box2):
    b1_x1 = box1[..., 0] - box1[..., 2] / 2
    b1_y1 = box1[..., 1] - box1[..., 3] / 2
    b1_x2 = box1[..., 0] + box1[..., 2] / 2
    b1_y2 = box1[..., 1] + box1[..., 3] / 2

    b2_x1 = box2[..., 0] - box2[..., 2] / 2
    b2_y1 = box2[..., 1] - box2[..., 3] / 2
    b2_x2 = box2[..., 0] + box2[..., 2] / 2
    b2_y2 = box2[..., 1] + box2[..., 3] / 2

    # Intersection
    inter_x1 = tf.maximum(b1_x1, b2_x1)
    inter_y1 = tf.maximum(b1_y1, b2_y1)
    inter_x2 = tf.minimum(b1_x2, b2_x2)
    inter_y2 = tf.minimum(b1_y2, b2_y2)

    inter_area = tf.maximum(0.0, inter_x2 - inter_x1) * tf.maximum(0.0, inter_y2 - inter_y1)

    # Areas
    area1 = tf.maximum(0.0, (b1_x2 - b1_x1) * (b1_y2 - b1_y1))
    area2 = tf.maximum(0.0, (b2_x2 - b2_x1) * (b2_y2 - b2_y1))

    # Union
    union = tf.maximum(area1 + area2 - inter_area, 1e-7)

    # IoU
    iou = inter_area / union

    # Enclosing box
    c_x1 = tf.minimum(b1_x1, b2_x1)
    c_y1 = tf.minimum(b1_y1, b2_y1)
    c_x2 = tf.maximum(b1_x2, b2_x2)
    c_y2 = tf.maximum(b1_y2, b2_y2)

    c_area = tf.maximum(0.0, (c_x2 - c_x1) * (c_y2 - c_y1))

    # GIoU
    giou = iou - (c_area - union) / (c_area + 1e-7)
    return giou


# ---------------------
# Focal Loss
# ---------------------
def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

    bce = -(y_true * tf.math.log(y_pred) +
            (1 - y_true) * tf.math.log(1 - y_pred))

    pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
    focal_weight = alpha * tf.pow(1 - pt, gamma)

    return focal_weight * bce


# ==========================================
# DETECTION LOSS CLASS
# ==========================================
# from tensorflow.keras import saving as keras_saving
# @keras.saving.register_keras_serializable()
class DetectionLoss(keras.losses.Loss):
    def __init__(self,
                 num_classes=10,
                 lambda_coord=5.0,
                 lambda_obj=1.0,
                 lambda_class=1.0,
                 lambda_noobj=0.5,
                 class_weights=None,
                 name="detection_loss"):

        super().__init__(name=name)

        self.num_classes = num_classes
        self.lambda_coord = lambda_coord
        self.lambda_obj = lambda_obj
        self.lambda_class = lambda_class
        self.lambda_noobj = lambda_noobj

        # Metrics
        self.giou_metric = keras.metrics.Mean(name="giou_loss")
        self.obj_metric = keras.metrics.Mean(name="obj_loss")
        self.noobj_metric = keras.metrics.Mean(name="noobj_loss")
        self.class_metric = keras.metrics.Mean(name="class_loss")

        # Class weights
        if class_weights is None:
            self.class_weights = tf.ones(num_classes, dtype=tf.float32)
        else:
            self.class_weights = tf.constant(list(class_weights.values()), dtype=tf.float32)

    @property
    def metrics(self):
        return [
            self.giou_metric,
            self.obj_metric,
            self.noobj_metric,
            self.class_metric
        ]

    def call(self, y_true, y_pred):
        # Predictions
        pred_xy = tf.sigmoid(y_pred[..., 0:2])
        pred_wh = y_pred[..., 2:4]
        pred_obj = tf.sigmoid(y_pred[..., 4:5])
        pred_class = tf.sigmoid(y_pred[..., 5:])

        # Ground truth
        true_xy = y_true[..., 0:2]
        true_wh = y_true[..., 2:4]
        true_obj = y_true[..., 4:5]
        true_class = y_true[..., 5:]

        obj_mask = true_obj
        noobj_mask = 1 - obj_mask

        # Box decoding
        pred_wh_c = tf.clip_by_value(pred_wh, -5.0, 5.0)
        true_wh_c = tf.clip_by_value(true_wh, -5.0, 5.0)

        pred_boxes = tf.concat([pred_xy, tf.exp(pred_wh_c)], axis=-1)
        true_boxes = tf.concat([true_xy, tf.exp(true_wh_c)], axis=-1)

        # GIoU loss
        giou = compute_giou(pred_boxes, true_boxes)[..., tf.newaxis]
        giou_loss = obj_mask * (1 - giou)
        giou_loss = tf.reduce_sum(giou_loss) / (tf.reduce_sum(obj_mask) + 1e-7)
        self.giou_metric.update_state(giou_loss)

        # Object loss
        obj_loss = obj_mask * focal_loss(true_obj, pred_obj)
        obj_loss = tf.reduce_sum(obj_loss) / (tf.reduce_sum(obj_mask) + 1e-7)
        self.obj_metric.update_state(obj_loss)

        # No-object loss
        noobj_loss = noobj_mask * focal_loss(true_obj, pred_obj)
        noobj_loss = tf.reduce_sum(noobj_loss) / (tf.reduce_sum(noobj_mask) + 1e-7)
        self.noobj_metric.update_state(noobj_loss)

        # Classification loss
        class_loss_raw = focal_loss(true_class, pred_class)
        cw = tf.reshape(self.class_weights, [1, 1, 1, 1, self.num_classes])
        class_loss_raw = class_loss_raw * cw * true_class
        class_loss_final = tf.reduce_sum(class_loss_raw) / (tf.reduce_sum(obj_mask) + 1e-7)
        self.class_metric.update_state(class_loss_final)

        # Total loss
        total_loss = (
            self.lambda_coord * giou_loss +
            self.lambda_obj * obj_loss +
            self.lambda_noobj * noobj_loss +
            self.lambda_class * class_loss_final
        )

        return total_loss


# ==========================================
# CREATE LOSS FUNCTION
# ==========================================

print("\nCreating loss function...")
loss_fn = DetectionLoss(
    num_classes=cfg.NUM_CLASSES,
    lambda_coord=5.0,
    lambda_obj=1.0,
    lambda_class=1.0,
    lambda_noobj=0.5,
    class_weights=cfg.CLASS_WEIGHTS
)

# ==========================================
# TEST LOSS FUNCTION
# ==========================================

print("\nTesting loss function...")
for images, grids in train_dataset.take(1):
    predictions = model(images, training=True)
    loss_value = loss_fn(grids, predictions)

print("  Loss computation successful")
print(f"  Total loss: {loss_value.numpy():.4f}")
print("\nLoss function ready")



Creating loss function...

Testing loss function...


Exception ignored in: <generator object create_dataset.<locals>.generator at 0x7f485f9a69e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/dataset_ops.py", line 872, in iterator_completed
    del self._iterators[self._normalize_id(iterator_id)]
        ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: generator ignored GeneratorExit


  Loss computation successful
  Total loss: 5.2382

Loss function ready


In [14]:
import time
from datetime import datetime

print("\nSetting up training...")

# Learning rate schedule
FIXED_LR = 5e-5

def create_lr_schedule(initial_lr=5e-5, warmup_epochs=5, total_epochs=150):
    def lr_schedule(epoch):
        if epoch < warmup_epochs:
            return initial_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        cosine_decay = 0.5 * (1 + np.cos(np.pi * progress))
        return 1e-6 + (initial_lr - 1e-6) * cosine_decay
    return lr_schedule

lr_callback = keras.callbacks.LearningRateScheduler(
    create_lr_schedule(FIXED_LR, 5, cfg.EPOCHS),
    verbose=0
)

# Setup callbacks
checkpoint_path = "/kaggle/working/FYP_Model_Checkpoints/best_model.keras"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

checkpoint = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True,
    verbose=1
)

log_dir = f"/kaggle/working/FYP_Logs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
tensorboard = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

csv_log = keras.callbacks.CSVLogger(
    "/kaggle/working/FYP_Logs/training.csv",
    append=True
)

# NaN detector
class NaNStopper(keras.callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        loss = logs.get('loss', 0)
        if np.isnan(loss) or np.isinf(loss):
            print(f"\nNaN detected at batch {batch}. Stopping training.")
            self.model.stop_training = True

# Epoch logger
class EpochLogger(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.start = time.time()
        print(f"\nEpoch {epoch+1}/{cfg.EPOCHS}")

    def on_epoch_end(self, epoch, logs=None):
        t = time.time() - self.start
        logs = logs or {}
        print(f"\nEpoch {epoch+1} complete ({t:.1f}s)")
        print(f"  Loss: {logs.get('loss', 0):.4f}")
        print(f"  Val Loss: {logs.get('val_loss', 0):.4f}")
        print(f"  LR: {logs.get('lr', 0):.6f}")

callbacks = [
    lr_callback,
    checkpoint,
    early_stop,
    tensorboard,
    csv_log,
    NaNStopper(),
    EpochLogger()
]

# Compile model
optimizer = keras.optimizers.Adam(
    learning_rate=FIXED_LR * 2,
    clipnorm=5.0,       # ⭐ FIXED — only ONE gradient clipping method allowed
    epsilon=1e-7
)

model.compile(optimizer=optimizer, loss=loss_fn, metrics=[])

print(f"\nTraining configuration:")
print(f"  Learning rate: {FIXED_LR}")
print(f"  Gradient clipping: clipnorm=5.0")
print(f"  Batch size: {cfg.BATCH_SIZE}")
print(f"  Epochs: {cfg.EPOCHS}")
print(f"  Checkpoints: {checkpoint_path}")

print("\nReady to start training")


Setting up training...

Training configuration:
  Learning rate: 5e-05
  Gradient clipping: clipnorm=5.0
  Batch size: 16
  Epochs: 150
  Checkpoints: /kaggle/working/FYP_Model_Checkpoints/best_model.keras

Ready to start training


In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
## CELL 14: Start Training
# python
# Start model training
print("\nStarting training...")
print("This will take several hours. Training progress will be saved automatically.")
print("You can stop training anytime with Ctrl+C")

try:
    history = model.fit(
        train_dataset,
        validation_data=valid_dataset,
        epochs=cfg.EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    print("\nTraining completed successfully")

    # Save final model
    final_path = "/kaggle/working/FYP_Model_Checkpoints/final_model.keras"
    model.save(final_path)
    print(f"\nFinal model saved to: {final_path}")

except KeyboardInterrupt:
    print("\nTraining interrupted by user")
    print("Best model has been saved")

except Exception as e:
    print(f"\nTraining error: {e}")
    print("Check saved checkpoints")

print("\nDone")





Starting training...
This will take several hours. Training progress will be saved automatically.
You can stop training anytime with Ctrl+C

Epoch 1/150
Epoch 1/150


I0000 00:00:1765551124.550321     131 service.cc:148] XLA service 0x7f47f000d750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765551124.551780     131 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1765551124.551805     131 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
2025-12-12 14:52:42.073897: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng0{} for conv (f32[16,32,641,641]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,64,320,320]{3,2,1,0}, f32[64,32,3,3]{3,2,1,0}), window={size=3x3 stride=2x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-12-12 14:52:43.

   2090/Unknown 1978s 906ms/step - loss: 4.1663

2025-12-12 15:25:06.388409: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng0{} for conv (f32[8,32,641,641]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,320,320]{3,2,1,0}, f32[64,32,3,3]{3,2,1,0}), window={size=3x3 stride=2x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-12-12 15:25:06.683208: E external/local_xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.294979381s
Trying algorithm eng0{} for conv (f32[8,32,641,641]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,64,320,320]{3,2,1,0}, f32[64,32,3,3]{3,2,1,0}), window={size=3x3 stride=2x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", backend_config={"cudnn_conv_backend_config":{"activation_

   2091/Unknown 2024s 928ms/step - loss: 4.1661

Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.



Epoch 1: val_loss improved from inf to 3.13383, saving model to /kaggle/working/FYP_Model_Checkpoints/best_model.keras

Epoch 1 complete (2089.1s)
  Loss: 3.8161
  Val Loss: 3.1338
  LR: 0.000000
2091/2091 ━━━━━━━━━━━━━━━━━━━━ 2089s 959ms/step - loss: 4.1660 - val_loss: 3.1338 - learning_rate: 1.0000e-05

Epoch 2/150
Epoch 2/150
1127/2091 ━━━━━━━━━━━━━━━━━━━━ 14:34 907ms/step - loss: 3.4054